# 🧠 DSA Lab 03: Arrays: Static Arrays, Dynamic Arrays and 2D Matrices
### Student Notebook

**Data Structures & Algorithms Lab Manual | University of Okara | Fall 2026**

Prepared by **Muhammad Ramzan**<br>
MPhil Scholar (Artificial Intelligence), PUCIT, University of the Punjab, Lahore<br>
Lab Engineer, Department of Computer Science, University of Okara

🔗 [GitHub](https://github.com/RamzanPUCIT) · [LinkedIn](https://www.linkedin.com/in/ramzan-ai) · [ResearchGate](https://www.researchgate.net/profile/Muhammad-Ramzan-105)

---
**How to use:** run each code cell with **Shift + Enter**, from top to bottom. Read the matching section in the Lab 03 manual, change values and re-run to see what happens.

> 📝 **Your job:** run the examples, then complete every `# TODO` in the Lab Tasks part. Each task has a **test cell**: when it prints ✅ your solution is correct. Write pseudocode as comments before coding!

## 📖 Part A: What Is an Array?

An **array** is a collection of elements of the **same type** stored in **contiguous (side-by-side) memory locations**. Each element is identified by an **index**, starting from **0**.

**An array in memory**

```text
Index:     0     1     2     3     4
        +-----+-----+-----+-----+-----+
marks:  | 67  | 89  | 45  | 92  | 78  |
        +-----+-----+-----+-----+-----+
Address: 1000  1004  1008  1012  1016     (each int = 4 bytes)
```

### Why Is Access O(1)? The Address Formula

Because elements are side by side and all have the same size, the computer can **calculate** the address of any element directly, without visiting the others:

> **📐 Address formula (1D)**
>
> **Address(A[i]) = Base address + i × size of each element**
> Example: base = 1000, each int is 4 bytes. Address of A[3] = 1000 + 3 × 4 = **1012**. One multiplication and one addition, no matter how big the array is. That is **O(1)**.

### Static vs Dynamic Arrays

|  | Static array | Dynamic array |
|---|---|---|
| Size | Fixed when created | Grows and shrinks automatically |
| Examples | C/C++ `int a[10];` | Python `list`, C++ `vector`, Java `ArrayList` |
| If full | Cannot add more elements | Allocates a bigger block and copies elements |
| Memory | Exactly what you asked for | Keeps some extra capacity for future growth |

## ⚙️ Part B: Array Operations and Their Cost

We will now implement each operation **by hand**, as if we had only a fixed-size block of memory. This shows what Python lists do internally.

### Traversal: Visit Every Element, O(n)

In [ ]:
marks = [67, 89, 45, 92, 78]
for i in range(len(marks)):
    print(f"marks[{i}] = {marks[i]}")

### Access and Update: O(1)

In [ ]:
marks = [67, 89, 45, 92, 78]
print(marks[3])      # access: O(1)
marks[3] = 95        # update: O(1)
print(marks)

### Insertion at a Position: O(n)

To insert at index `pos`, every element from `pos` to the end must **shift one place right** to make room. In the worst case (inserting at index 0) all n elements move.

**Dry run**

```text
Insert 50 at index 1 in [67, 89, 45, 92, _ ]

Step 1: shift 92 right   -> [67, 89, 45, _ , 92]
Step 2: shift 45 right   -> [67, 89, _ , 45, 92]
Step 3: shift 89 right   -> [67, _ , 89, 45, 92]
Step 4: place 50         -> [67, 50, 89, 45, 92]
```

In [ ]:
def insert_at(arr, size, pos, value):
    """arr has fixed capacity; size = number of used slots."""
    if size == len(arr):
        raise OverflowError("Array is full")
    for i in range(size, pos, -1):   # shift right, starting from the end
        arr[i] = arr[i - 1]
    arr[pos] = value
    return size + 1

arr = [67, 89, 45, 92, None]         # capacity 5, size 4
size = insert_at(arr, 4, 1, 50)
print(arr, "size =", size)

### Deletion at a Position: O(n)

To delete the element at index `pos`, every element after it **shifts one place left** to fill the gap.

In [ ]:
def delete_at(arr, size, pos):
    for i in range(pos, size - 1):   # shift left
        arr[i] = arr[i + 1]
    arr[size - 1] = None             # clear the last used slot
    return size - 1

arr = [67, 50, 89, 45, 92]
size = delete_at(arr, 5, 1)
print(arr, "size =", size)

### Linear Search: O(n)

In [ ]:
def linear_search(arr, target):
    for i in range(len(arr)):
        if arr[i] == target:
            return i                 # found: return the index
    return -1                        # not found

print(linear_search([67, 89, 45, 92, 78], 92), linear_search([1, 2], 7))

| Operation | Time | Why |
|---|---|---|
| Access / update `a[i]` | O(1) | Address formula |
| Traverse | O(n) | Visit every element |
| Search (unsorted) | O(n) | May check every element |
| Insert / delete at end | O(1) | No shifting needed |
| Insert / delete at beginning or middle | O(n) | Elements must shift |

## 🧱 Part C: Python Classes in 5 Minutes

From now on we will build our own data structures. In Python, we do this with a **class**. A class is a **blueprint**; an object is a thing built from that blueprint.

> **🏠 Analogy**
>
> A class is like the **map of a house**. Using one map you can build many houses (objects). Each house has its own rooms (attributes) and things you can do in it (methods).

In [ ]:
class Student:
    def __init__(self, name, roll):     # constructor: runs when an object is created
        self.name = name                # attribute
        self.roll = roll
        self.marks = []

    def add_mark(self, m):              # method
        self.marks.append(m)

    def average(self):
        return sum(self.marks) / len(self.marks) if self.marks else 0

    def __str__(self):                  # how the object looks when printed
        return f"{self.roll}: {self.name} (avg {self.average():.1f})"

s = Student("Ayesha", "BSCS-21")
s.add_mark(80); s.add_mark(90)
print(s)

`self` refers to the object itself. `__init__` sets up the object, and `__str__` decides how it prints. We will use special methods like `__len__` and `__getitem__` so our structures behave like Python lists.

## 📦 Part D: Build Your Own Dynamic Array

How does a Python list grow without limit if memory blocks are fixed? Answer: when the block is full, it **allocates a new block (usually about double the size), copies all elements, and frees the old block**.

**Resizing by doubling**

```text
Capacity 2: [10, 20]          append 30 -> FULL!
New capacity 4: copy 10, 20  -> [10, 20, 30, _ ]
append 40                    -> [10, 20, 30, 40]    append 50 -> FULL!
New capacity 8: copy 4 items -> [10, 20, 30, 40, 50, _, _, _]
```

In [ ]:
class DynamicArray:
    def __init__(self):
        self.capacity = 1
        self.size = 0
        self.data = [None] * self.capacity     # our fixed block of memory

    def __len__(self):
        return self.size

    def __getitem__(self, i):
        if not 0 <= i < self.size:
            raise IndexError("index out of range")
        return self.data[i]

    def _resize(self, new_capacity):
        new_data = [None] * new_capacity
        for i in range(self.size):             # copy: O(n)
            new_data[i] = self.data[i]
        self.data = new_data
        self.capacity = new_capacity

    def append(self, value):
        if self.size == self.capacity:
            self._resize(2 * self.capacity)    # double when full
        self.data[self.size] = value
        self.size += 1

    def __str__(self):
        return str([self.data[i] for i in range(self.size)]) + f" (capacity {self.capacity})"

arr = DynamicArray()
for x in [10, 20, 30, 40, 50]:
    arr.append(x)
    print(arr)

### Amortised O(1): Why Doubling Is Clever

A resize costs O(n), but it happens rarely. If we append n items, the total copying work is 1 + 2 + 4 + … + n/2 + n < **2n**. Spread over n appends, that is less than **2 copies per append on average**, so each append is **amortised O(1)**.

In [ ]:
class CountingArray(DynamicArray):
    copies = 0
    def _resize(self, new_capacity):
        CountingArray.copies += self.size
        super()._resize(new_capacity)

for n in [1_000, 10_000, 100_000]:
    CountingArray.copies = 0
    a = CountingArray()
    for i in range(n):
        a.append(i)
    print(f"n = {n:>7,}: total copies = {CountingArray.copies:>7,}  ({CountingArray.copies / n:.2f} per append)")

> **🤔 What if we grew by +1 instead of ×2?**
>
> Then every append would trigger a copy, and appending n items would cost 1 + 2 + … + n = **O(n²)**. Doubling turns that into O(n) total. A small design decision with a huge impact.

## 🔲 Part E: 2D Arrays (Matrices)

A **2D array** stores data in **rows and columns**. In Python we represent it as a **list of lists**.

In [ ]:
#            Maths  Physics  CS
result = [ [ 78,     85,     90 ],   # row 0: Ali
           [ 66,     72,     95 ],   # row 1: Sara
           [ 88,     91,     79 ] ]  # row 2: Hamza

print(result[1][2])                  # Sara's CS marks -> 95
rows, cols = len(result), len(result[0])

for r in range(rows):                # row-wise traversal
    print("Row", r, "total =", sum(result[r]))

for c in range(cols):                # column-wise traversal
    col_total = 0
    for r in range(rows):
        col_total += result[r][c]
    print("Subject", c, "average =", round(col_total / rows, 1))

> **⚠️ The classic 2D trap**
>
> `grid = [[0] * 3] * 3` creates **three references to the same row**. Changing `grid[0][0]` changes all rows! Always use `grid = [[0] * 3 for _ in range(3)]`.

In [ ]:
wrong = [[0] * 3] * 3
wrong[0][0] = 9
print("wrong:", wrong)

right = [[0] * 3 for _ in range(3)]
right[0][0] = 9
print("right:", right)

### Row-Major Storage and the 2D Address Formula

Memory is one-dimensional, so a 2D array is stored **row after row** (row-major order, used by C, C++ and NumPy by default).

> **📐 Address formula (2D, row-major)**
>
> **Address(A[i][j]) = Base + (i × number_of_columns + j) × element_size**
> Example: a 3 × 4 int matrix at base 2000. Address of A[2][1] = 2000 + (2 × 4 + 1) × 4 = **2036**.

### Transpose and Matrix Multiplication

In [ ]:
def transpose(m):
    rows, cols = len(m), len(m[0])
    t = [[0] * rows for _ in range(cols)]
    for i in range(rows):
        for j in range(cols):
            t[j][i] = m[i][j]
    return t

def mat_mult(a, b):                  # (n x m) * (m x p) -> (n x p): O(n*m*p)
    n, m, p = len(a), len(b), len(b[0])
    c = [[0] * p for _ in range(n)]
    for i in range(n):
        for j in range(p):
            for k in range(m):
                c[i][j] += a[i][k] * b[k][j]
    return c

A = [[1, 2, 3], [4, 5, 6]]
print(transpose(A))
print(mat_mult([[1, 2], [3, 4]], [[5, 6], [7, 8]]))

## ⚙️ C++ Corner

**C++**

```cpp
#include <iostream>
#include <vector>
using namespace std;

int main() {
    int marks[5] = {67, 89, 45, 92, 78};          // static array: fixed size
    cout << "Address of marks[0]: " << &marks[0] << endl;
    cout << "Address of marks[1]: " << &marks[1] << endl;   // 4 bytes later

    vector<int> v;                                 // dynamic array
    for (int i = 0; i < 10; i++) {
        v.push_back(i);
        cout << "size=" << v.size() << " capacity=" << v.capacity() << endl;
    }

    int grid[3][4] = {0};                          // 2D static array
    grid[2][1] = 7;
    cout << grid[2][1] << endl;
    return 0;
}
```

## 🧪 Lab Tasks
Difficulty: 🟢 Easy · 🟡 Medium · 🔴 Hard

### Task 1 🟢 Array Statistics in One Pass
Write `stats(arr)` that returns `(minimum, maximum, average)` using **a single loop** and no built-in `min`, `max` or `sum`. Return `None` for an empty array.

```text
stats([67, 89, 45, 92, 78]) → (45, 92, 74.2)
```

In [ ]:
def stats(arr):
    # TODO: one loop, no min/max/sum
    pass

In [ ]:
# Test cell: run after your solution
assert stats([67, 89, 45, 92, 78]) == (45, 92, 74.2)
assert stats([5]) == (5, 5, 5.0)
assert stats([]) is None
assert stats([-3, -1, -2]) == (-3, -1, -2.0)
print("✅ Task 1 passed")

### Task 2 🟢 Move Zeroes to the End
Move all `0`s to the end of the array **in place** while keeping the order of the non-zero elements. Use O(1) extra space. Return the same list.

```text
[0, 1, 0, 3, 12] → [1, 3, 12, 0, 0]
```

In [ ]:
def move_zeroes(arr):
    write = 0
    # TODO: in-place, O(n) time, O(1) space
    return arr

In [ ]:
# Test cell: run after your solution
assert move_zeroes([0, 1, 0, 3, 12]) == [1, 3, 12, 0, 0]
assert move_zeroes([0, 0]) == [0, 0]
assert move_zeroes([1, 2]) == [1, 2]
a = [4, 0, 5]; assert move_zeroes(a) is a
print("✅ Task 2 passed")

### Task 3 🟡 Rotate an Array
Rotate the array to the right by `k` steps **in place** (k may be larger than n).

Hint (reversal trick): reverse the whole array, then reverse the first k elements, then reverse the rest. This is O(n) time and O(1) space.

```text
rotate([1, 2, 3, 4, 5, 6, 7], 3) → [5, 6, 7, 1, 2, 3, 4]
```

In [ ]:
def reverse_part(arr, left, right):
    # TODO: reverse arr[left..right] in place
    pass

def rotate(arr, k):
    # TODO: use reverse_part three times
    return arr

In [ ]:
# Test cell: run after your solution
assert rotate([1, 2, 3, 4, 5, 6, 7], 3) == [5, 6, 7, 1, 2, 3, 4]
assert rotate([1, 2], 5) == [2, 1]
assert rotate([], 3) == []
assert rotate([1, 2, 3], 0) == [1, 2, 3]
print("✅ Task 3 passed")

### Task 4 🟡 Merge Two Sorted Arrays
Two sections’ merit lists are each sorted in ascending order. Write `merge_sorted(a, b)` that returns one sorted list in **O(n + m)** time. Do not use `sort()` or `sorted()`. (This idea is the heart of Merge Sort in Lab 07.)

```text
merge_sorted([1, 4, 9], [2, 3, 10, 12]) → [1, 2, 3, 4, 9, 10, 12]
```

In [ ]:
def merge_sorted(a, b):
    i = j = 0
    result = []
    # TODO: compare a[i] and b[j], take the smaller one
    # TODO: copy whatever is left
    return result

In [ ]:
# Test cell: run after your solution
assert merge_sorted([1, 4, 9], [2, 3, 10, 12]) == [1, 2, 3, 4, 9, 10, 12]
assert merge_sorted([], [1, 2]) == [1, 2]
assert merge_sorted([5], []) == [5]
assert merge_sorted([1, 1], [1]) == [1, 1, 1]
print("✅ Task 4 passed")

### Task 5 🟡 Complete the Dynamic Array
Extend the `DynamicArray` class from Part D (redefined in the notebook) with these methods:

`insert(pos, value)` (shift right, resize if full), `remove_at(pos)` (shift left and return the removed value), and `index_of(value)` (return the index or −1). Raise `IndexError` for invalid positions.

In [ ]:
class MyArray(DynamicArray):
    def insert(self, pos, value):
        # TODO: validate pos, resize if full, shift right, place value
        pass

    def remove_at(self, pos):
        # TODO: validate pos, save value, shift left, shrink size
        pass

    def index_of(self, value):
        # TODO: linear search
        pass

In [ ]:
# Test cell: run after your solution
a = MyArray()
for x in [10, 20, 30]:
    a.append(x)
a.insert(0, 5); a.insert(2, 15); a.insert(5, 40)
assert [a[i] for i in range(len(a))] == [5, 10, 15, 20, 30, 40]
assert a.remove_at(2) == 15 and len(a) == 5
assert a.index_of(30) == 3 and a.index_of(99) == -1
try:
    a.insert(10, 1); raise AssertionError("should raise IndexError")
except IndexError:
    pass
print("✅ Task 5 passed")

### Task 6 🔴 Rotate an Image by 90°
A grayscale image is an n × n matrix. Rotate it **90° clockwise in place**.

Hint: transpose the matrix in place (swap `m[i][j]` with `m[j][i]` for j > i), then reverse each row.

```text
[[1, 2, 3],        [[7, 4, 1],
 [4, 5, 6],   →    [8, 5, 2],
 [7, 8, 9]]         [9, 6, 3]]
```

In [ ]:
def rotate_image(m):
    n = len(m)
    # TODO 1: transpose in place
    # TODO 2: reverse each row
    return m

In [ ]:
# Test cell: run after your solution
assert rotate_image([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [[7, 4, 1], [8, 5, 2], [9, 6, 3]]
assert rotate_image([[1]]) == [[1]]
assert rotate_image([[1, 2], [3, 4]]) == [[3, 1], [4, 2]]
print("✅ Task 6 passed")

### Task 7 🔴 Spiral Order of a Matrix
Return all elements of an m × n matrix in **spiral order** (clockwise from the top-left). Keep four boundaries: top, bottom, left, right, and shrink them after each side.

```text
[[1, 2, 3],
 [4, 5, 6],   → [1, 2, 3, 6, 9, 8, 7, 4, 5]
 [7, 8, 9]]
```

In [ ]:
def spiral(m):
    if not m:
        return []
    top, bottom, left, right = 0, len(m) - 1, 0, len(m[0]) - 1
    out = []
    # TODO: walk right, down, left, up while shrinking the boundaries
    return out

In [ ]:
# Test cell: run after your solution
assert spiral([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [1, 2, 3, 6, 9, 8, 7, 4, 5]
assert spiral([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]) == [1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7]
assert spiral([[7]]) == [7]
assert spiral([[1], [2], [3]]) == [1, 2, 3]
assert spiral([]) == []
print("✅ Task 7 passed")

## 🔬 Bonus: Arrays in AI and Remote Sensing with NumPy

In [ ]:
# Bonus: run after completing the tasks
import numpy as np

# A tiny "satellite image" with two bands (values 0-1)
red = np.array([[0.20, 0.30, 0.25], [0.10, 0.40, 0.35], [0.05, 0.15, 0.30]])
nir = np.array([[0.60, 0.55, 0.30], [0.70, 0.45, 0.40], [0.80, 0.65, 0.35]])

ndvi = (nir - red) / (nir + red)          # element-wise on whole arrays, no loops!
print("NDVI (vegetation index):")
print(np.round(ndvi, 2))
print("Healthy vegetation pixels (NDVI > 0.5):", int((ndvi > 0.5).sum()))

# One neural-network layer = matrix multiplication
x = np.array([[1.0, 2.0, 3.0]])           # input (1 x 3)
W = np.array([[0.1, 0.2], [0.3, 0.4], [0.5, 0.6]])  # weights (3 x 2)
print("Layer output:", x @ W)

---
## 🎉 Lab 03 Complete!

Next: **Lab 04: Searching Techniques (Linear, Binary, Two Pointers, Sliding Window)**

✅ Upload this notebook to your GitHub repository `DSA-Labs/Lab03/`.

---
*Prepared by **Muhammad Ramzan**, MPhil Scholar (AI), PUCIT, University of the Punjab*<br>
🔗 [GitHub](https://github.com/RamzanPUCIT) · [LinkedIn](https://www.linkedin.com/in/ramzan-ai) · [ResearchGate](https://www.researchgate.net/profile/Muhammad-Ramzan-105)